In [1]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql.functions import col, sum as spark_sum, when


In [2]:

ROOT_DIR = Path.cwd().resolve().parent


path_parquet = ROOT_DIR / "data" / "bronze" / "yellow_tripdata_2025-01.parquet"


spark = SparkSession.builder.appName("taxi").getOrCreate()

taxi_trips_data = spark.read.parquet(str(path_parquet))

taxi_trips_data.show(5)
taxi_trips_data.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [3]:
taxi_trips_data.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [4]:
total_rows = taxi_trips_data.count()

null_counts = taxi_trips_data.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in taxi_trips_data.columns
])

null_report = null_counts.toPandas().T.reset_index()
null_report.columns = ["column_name", "null_count"]

null_report["null_percentage"] = (
    null_report["null_count"] / total_rows * 100
).round(2)

null_report = null_report.sort_values(
    by="null_percentage",
    ascending=False
)
print(f"The total rows is: {total_rows}")
null_report

The total rows is: 3475226


,column_name,null_count,null_percentage
3,passenger_count,540149,15.54
17,congestion_surcharge,540149,15.54
6,store_and_fwd_flag,540149,15.54
5,RatecodeID,540149,15.54
18,Airport_fee,540149,15.54
2,tpep_dropoff_datetime,0,0.00
0,VendorID,0,0.00
1,tpep_pickup_datetime,0,0.00
8,DOLocationID,0,0.00
9,payment_type,0,0.00


In [ ]:

total = taxi_trips_data.count()
null_columns_list = ["passenger_count", "congestion_surcharge","store_and_fwd_flag","RatecodeID","Airport_fee"]
for col_name in null_columns_list:
  taxi_trips_data.groupBy(col_name) \
    .count() \
    .withColumn("percentage", (col("count") / total) * 100) \
    .orderBy(col("count").desc()) \
    .show()

+---------------+-------+--------------------+
|passenger_count|  count|          percentage|
+---------------+-------+--------------------+
|              1|2322434|   66.82828685098465|
|           NULL| 540149|   15.54284527107014|
|              2| 407761|  11.733366405522979|
|              3|  91409|   2.630303755784516|
|              4|  59009|  1.6979902889768896|
|              0|  24656|  0.7094790381978036|
|              5|  17786|  0.5117940531061865|
|              6|  12004|  0.3454163844308255|
|              8|     11|3.165261770025891...|
|              7|      4|1.151004280009415...|
|              9|      3|8.632532100070614E-5|
+---------------+-------+--------------------+

+--------------------+-------+------------------+
|congestion_surcharge|  count|        percentage|
+--------------------+-------+------------------+
|                 2.5|2660818|  76.5653226581523|
|                NULL| 540149| 15.54284527107014|
|                 0.0| 225938|6.501390125419

In [16]:
taxi_trips_data.describe().show()

+-------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|          VendorID|   passenger_count|    trip_distance|       RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+-------

In [ ]:

null_columns_list = [("passenger_count",-1), 
                     ("congestion_surcharge",0),
                     ("store_and_fwd_flag","U"),
                     ("RatecodeID",99),
                     ("Airport_fee",0)]

df_with_flags = taxi_trips_data.withColumn(
    "passenger_count_was_null",
    col("passenger_count").isNull()
)

taxi_trips_data = taxi_trips_data.fillna({
    "passenger_count": -1,
    "congestion_surcharge": 0,
    "store_and_fwd_flag": "U",
    "RatecodeID": 99,
    "Airport_fee": 0
})

{"ts": "2026-04-29 10:07:23.051", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`extra`, `mta_tax`, `VendorID`, `RatecodeID`, `tip_amount`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor44.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o31.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`extra`, `mta_tax`, `VendorID`, `RatecodeID`, `tip_amount`]. SQLSTATE: 42703;\n'Project [']\n+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,st

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`extra`, `mta_tax`, `VendorID`, `RatecodeID`, `tip_amount`]. SQLSTATE: 42703;
'Project [']
+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18,cbd_congestion_fee#19] parquet
